# 64×64 Streaming 2D Convolution on PYNQ

This notebook tests the `conv2d_stream` HLS accelerator for a **64×64 image** and a **3×3 kernel**.

The accelerator performs full convolution using a streaming AXI-DMA input/output path. Kernel coefficients are written once through AXI-Lite registers before streaming begins.

Expected dimensions:

```text
Input image      : 64 × 64
Kernel           : 3 × 3
Output image     : 66 × 66
DMA input words  : 4224 = 66 × 64 = 4096 real pixels + 128 trailing zeros
DMA output words : 4356 = 66 × 66
```

The HLS implementation assumes the kernel is already in the sliding-window orientation. If mathematical convolution is desired, provide the already-flipped kernel.


## 1. Imports and overlay loading

Place `conv2d_design.bit` and `conv2d_design.hwh` in the same directory as this notebook before running it.


In [1]:
import time
import numpy as np
from pynq import Overlay, allocate


In [2]:
ol = Overlay("conv2d_design.bit")
print("Overlay loaded successfully")
print("IP blocks:")
for name in ol.ip_dict.keys():
    print("  ", name)

# Expected handles from the Vivado block design
dma = ol.axi_dma_0
conv2d = ol.conv2d_stream_0
hw_timer = ol.axi_timer_0


Overlay loaded successfully
IP blocks:
   conv2d_stream_0
   axi_timer_0
   axi_dma_0
   ps7


## 2. Dimensions

These constants must match the HLS header file.


In [3]:
IH = 64
IW = 64
KH = 3
KW = 3

OH = IH + KH - 1   # 66
OW = IW + KW - 1   # 66

N_INPUT_REAL = IH * IW          # 4096 real image pixels
N_INPUT_STREAM = OH * IW        # 66 * 64 = 4224, because current HLS reads two extra zero rows
N_OUTPUT = OH * OW              # 66 * 66 = 4356

print(f"Input image shape     : {IH} x {IW}")
print(f"Kernel shape          : {KH} x {KW}")
print(f"Output image shape    : {OH} x {OW}")
print(f"DMA input words       : {N_INPUT_STREAM}")
print(f"DMA output words      : {N_OUTPUT}")


Input image shape     : 64 x 64
Kernel shape          : 3 x 3
Output image shape    : 66 x 66
DMA input words       : 4224
DMA output words      : 4356


## 3. AXI-Lite register map

These offsets were found from the generated HLS driver header. Use:

```bash
grep -n "k00\|k01\|k02\|k10\|k11\|k12\|k20\|k21\|k22" \
  hls_2dconv_stream_pipelined/sol1/impl/ip/drivers/conv2d_stream_v1_0/src/xconv2d_stream_hw.h
```


In [4]:
CTRL_REG = 0x00

K_OFFSETS = {
    "k00": 0x10,
    "k01": 0x18,
    "k02": 0x20,
    "k10": 0x28,
    "k11": 0x30,
    "k12": 0x38,
    "k20": 0x40,
    "k21": 0x48,
    "k22": 0x50,
}

def write_kernel(ip, kernel):
    """Write a 3x3 integer kernel to AXI-Lite registers."""
    kernel = np.asarray(kernel, dtype=np.int32)
    assert kernel.shape == (3, 3)

    ip.write(K_OFFSETS["k00"], int(kernel[0, 0]))
    ip.write(K_OFFSETS["k01"], int(kernel[0, 1]))
    ip.write(K_OFFSETS["k02"], int(kernel[0, 2]))

    ip.write(K_OFFSETS["k10"], int(kernel[1, 0]))
    ip.write(K_OFFSETS["k11"], int(kernel[1, 1]))
    ip.write(K_OFFSETS["k12"], int(kernel[1, 2]))

    ip.write(K_OFFSETS["k20"], int(kernel[2, 0]))
    ip.write(K_OFFSETS["k21"], int(kernel[2, 1]))
    ip.write(K_OFFSETS["k22"], int(kernel[2, 2]))

def start_ip(ip):
    """Start the HLS IP."""
    ip.write(CTRL_REG, 0x01)

def wait_ip_done(ip, timeout_s=1.0):
    """Optional polling of ap_done. DMA waits are usually enough."""
    t0 = time.perf_counter()
    while (ip.read(CTRL_REG) & 0x02) == 0:
        if time.perf_counter() - t0 > timeout_s:
            raise TimeoutError("conv2d_stream did not assert ap_done")
    return True


## 4. Python reference model

This matches the HLS implementation: the kernel is already in the orientation used by the sliding-window hardware, so the reference does **not** flip the kernel again.


In [5]:
def conv2d_full_preflipped_kernel(image, kernel):
    """
    Reference model matching the HLS implementation.

    The kernel is assumed to already be in the orientation used by
    the sliding-window hardware. This function does not flip it again.
    """
    image = np.asarray(image, dtype=np.int32)
    kernel = np.asarray(kernel, dtype=np.int32)

    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = ih + kh - 1
    ow = iw + kw - 1

    # For a 3x3 kernel, the HLS line buffer convention corresponds to a
    # 2-pixel zero border around the image in this reference model.
    padded = np.zeros((oh + 2, ow + 2), dtype=np.int32)
    padded[2:2+ih, 2:2+iw] = image

    out = np.zeros((oh, ow), dtype=np.int32)
    for r in range(oh):
        for c in range(ow):
            window = padded[r:r+kh, c:c+kw]
            out[r, c] = np.sum(window * kernel)

    return out


## 5. Prepare input image, kernel, and expected output

To keep the notebook readable, only small slices of the 64×64 image and 66×66 output are printed.


In [6]:
# Deterministic 64x64 test image
image = np.arange(1, IH * IW + 1, dtype=np.int32).reshape(IH, IW)

# Kernel loaded into HLS. This is assumed to be in sliding-window orientation.
kernel = np.array([
    [-1, -2, -3],
    [-4, -5, -6],
    [-7, -8, -9],
], dtype=np.int32)

expected = conv2d_full_preflipped_kernel(image, kernel)

print("Input image shape:", image.shape)
print("Top-left 5x5 of input image:")
print(image[:5, :5])

print("Kernel loaded into HLS:")
print(kernel)

print("Expected output shape:", expected.shape)
print("Top-left 8x8 of expected output:")
print(expected[:8, :8])


Input image shape: (64, 64)
Top-left 5x5 of input image:
[[  1   2   3   4   5]
 [ 65  66  67  68  69]
 [129 130 131 132 133]
 [193 194 195 196 197]
 [257 258 259 260 261]]
Kernel loaded into HLS:
[[-1 -2 -3]
 [-4 -5 -6]
 [-7 -8 -9]]
Expected output shape: (66, 66)
Top-left 8x8 of expected output:
[[    -9    -26    -50    -74    -98   -122   -146   -170]
 [  -591  -1131  -1618  -1657  -1696  -1735  -1774  -1813]
 [ -1554  -2931  -4128  -4173  -4218  -4263  -4308  -4353]
 [ -2706  -5043  -7008  -7053  -7098  -7143  -7188  -7233]
 [ -3858  -7155  -9888  -9933  -9978 -10023 -10068 -10113]
 [ -5010  -9267 -12768 -12813 -12858 -12903 -12948 -12993]
 [ -6162 -11379 -15648 -15693 -15738 -15783 -15828 -15873]
 [ -7314 -13491 -18528 -18573 -18618 -18663 -18708 -18753]]


## 6. Allocate DMA buffers

The current HLS implementation reads `OH × IW = 4224` input words. The first 4096 words are the real image pixels, and the final 128 words are zeros to complete the convolution (since the last output corresponds to the kernel overlapping with the last pixel of the image + padded zeros)


In [7]:
in_buf = allocate(shape=(N_INPUT_STREAM,), dtype=np.int32)
out_buf = allocate(shape=(N_OUTPUT,), dtype=np.int32)

in_buf[:N_INPUT_REAL] = image.flatten()
in_buf[N_INPUT_REAL:] = 0
out_buf[:] = 0

print(f"Input buffer shape : {in_buf.shape}")
print(f"Output buffer shape: {out_buf.shape}")
print("Last 8 input words, should be zeros:", np.array(in_buf[-8:]))


Input buffer shape : (4224,)
Output buffer shape: (4356,)
Last 8 input words, should be zeros: [0 0 0 0 0 0 0 0]


## 7. Run the accelerator using AXI DMA

Safe order:

1. Program kernel coefficients over AXI-Lite
2. Start DMA receive channel
3. Start DMA send channel
4. Start the HLS IP
5. Wait for both DMA channels


In [8]:
print("--- DMA Streaming 2D Convolution ---")

write_kernel(conv2d, kernel)

t_start = time.perf_counter()

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)
start_ip(conv2d)

dma.sendchannel.wait()
dma.recvchannel.wait()

t_dma = time.perf_counter() - t_start

hw_out = np.array(out_buf, dtype=np.int32).reshape(OH, OW)

print(f"DMA transfer + accelerator time: {t_dma * 1e3:.3f} ms")
print("Hardware output shape:", hw_out.shape)
print("Top-left 8x8 of hardware output:")
print(hw_out[:8, :8])


--- DMA Streaming 2D Convolution ---
DMA transfer + accelerator time: 6.084 ms
Hardware output shape: (66, 66)
Top-left 8x8 of hardware output:
[[    -9    -26    -50    -74    -98   -122   -146   -170]
 [  -591  -1131  -1618  -1657  -1696  -1735  -1774  -1813]
 [ -1554  -2931  -4128  -4173  -4218  -4263  -4308  -4353]
 [ -2706  -5043  -7008  -7053  -7098  -7143  -7188  -7233]
 [ -3858  -7155  -9888  -9933  -9978 -10023 -10068 -10113]
 [ -5010  -9267 -12768 -12813 -12858 -12903 -12948 -12993]
 [ -6162 -11379 -15648 -15693 -15738 -15783 -15828 -15873]
 [ -7314 -13491 -18528 -18573 -18618 -18663 -18708 -18753]]


## 8. Verify correctness

The hardware output should exactly match the integer Python reference model.


In [9]:
diff = hw_out - expected
max_abs_diff = np.max(np.abs(diff))

print("Difference shape:", diff.shape)
print("Top-left 8x8 of difference:")
print(diff[:8, :8])

print(f"Max absolute difference = {max_abs_diff}")

if max_abs_diff == 0:
    print("PASS: hardware output matches Python reference")
else:
    print("FAIL: hardware output differs from Python reference")


Difference shape: (66, 66)
Top-left 8x8 of difference:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]
Max absolute difference = 0
PASS: hardware output matches Python reference


## 9. AXI Timer measurement

This section reruns the accelerator and measures elapsed cycles using the AXI Timer IP.


In [10]:
TCSR0 = 0x00
TLR0  = 0x04
TCR0  = 0x08

def timer_start(timer):
    timer.write(TLR0, 0)
    timer.write(TCSR0, 0x020)   # LOAD
    timer.write(TCSR0, 0x080)   # ENT, count up

def timer_stop(timer):
    cycles = timer.read(TCR0)
    timer.write(TCSR0, 0x000)
    return cycles

# Reinitialize buffers
in_buf[:N_INPUT_REAL] = image.flatten()
in_buf[N_INPUT_REAL:] = 0
out_buf[:] = 0

write_kernel(conv2d, kernel)

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)

timer_start(hw_timer)
start_ip(conv2d)

dma.sendchannel.wait()
dma.recvchannel.wait()

cycles = timer_stop(hw_timer)

FCLK_MHZ = 100.0
hw_time_us = cycles / FCLK_MHZ

print(f"HW timer: {cycles} cycles = {hw_time_us:.3f} us @ {FCLK_MHZ:.0f} MHz")


HW timer: 190972 cycles = 1909.720 us @ 100 MHz


# Timing Scipy Convolution

Since the hardware version assumes the kernel is already in sliding-window orientation, we've to use `correlate2d` instead of `convolve2d` to avoid flipping the kernel again.

In [12]:
from scipy.signal import correlate2d
import time

print("\n--- SciPy correlate2d Timing ---")

N_RUNS = 1000

# Full output shape will be 66x66 for 64x64 input and 3x3 kernel.
y_scipy = correlate2d(image, kernel, mode="full", boundary="fill", fillvalue=0)

max_diff_scipy = np.max(np.abs(y_scipy - expected))
print(f"Max |SciPy correlate2d output - expected| = {max_diff_scipy}")

_ = correlate2d(image, kernel, mode="full", boundary="fill", fillvalue=0)

t_start = time.perf_counter()

for _ in range(N_RUNS):
    y_scipy = correlate2d(image, kernel, mode="full", boundary="fill", fillvalue=0)

t_total = time.perf_counter() - t_start
t_one = t_total / N_RUNS

print(f"Runs: {N_RUNS}")
print(f"Average SciPy correlate2d time: {t_one * 1e3:.6f} ms per convolution")

print("\n--- FPGA vs SciPy Timing ---")
print(f"FPGA DMA+accelerator time: {hw_time_us / 1000:.6f} ms")
print(f"SciPy correlate2d time:    {t_one * 1e3:.6f} ms")

speedup = (t_one * 1e6) / hw_time_us
print(f"FPGA speedup vs SciPy:     {speedup:.3f}x")


--- SciPy correlate2d Timing ---
Max |SciPy correlate2d output - expected| = 0
Runs: 1000
Average SciPy correlate2d time: 2.379813 ms per convolution

--- FPGA vs SciPy Timing ---
FPGA DMA+accelerator time: 1.909720 ms
SciPy correlate2d time:    2.379813 ms
FPGA speedup vs SciPy:     1.246x


## 10. Final verification after timed run


In [11]:
hw_out_timer = np.array(out_buf, dtype=np.int32).reshape(OH, OW)
max_abs_diff_timer = np.max(np.abs(hw_out_timer - expected))

print(f"Max absolute difference after timed run = {max_abs_diff_timer}")
if max_abs_diff_timer == 0:
    print("PASS: timed-run hardware output matches Python reference")
else:
    print("FAIL: timed-run hardware output differs from Python reference")


Max absolute difference after timed run = 0
PASS: timed-run hardware output matches Python reference


## 11. Cleanup

Run this cell when finished with the DMA buffers.


In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("DMA buffers freed")
